# Hindi Word-Order Replication -- full pipeline + trigram training reference, live on Google Drive

Same pipeline as `run_full_pipeline_on_drive.ipynb` -- raw HDTB treebank -> filtering -> word-order variant generation -> feature scoring -> pairwise classifier -> every paper table and figure, all computed live, cell by cell. This version additionally shows the **actual trigram-LM training code**, copied from `notebooks/MTP_prject_folder_structure.ipynb` (where it really ran on Colab), as a non-executed reference block in Step 4 -- so the training process is visible without this run needing KenLM, a 700 MB Wikipedia dump, or the ~1.4 GB of model files it produces.

1. **Load** the raw UD_Hindi-HDTB treebank (live)
2. **Filter** sentences to the paper's criteria (live)
3. **Generate** non-canonical word-order variants (live -- reproduces the exact same totals as the cached dataset, as a correctness check)
4. **Score** each variant's 5 features -- dependency length is live; trigram / PCFG / adaptive-LSTM / XGLM-1.7B / Gemma-3 surprisal are loaded from cache, with an explicit cell explaining why each one is cached rather than recomputed in Colab (trigram's actual training code is included as a reference, not executed)
5. **Classify**: the pairwise logistic-regression classifier (10-fold CV), defined and run live
6. **Report**: every paper table built from that run's live output, and every paper figure plotted live with matplotlib right after the table it's drawn from

**Files needed**, ~92 MB total:
```
src/                          (the pipeline's library code -- loader, filter, variant generator)
data/ud_hindi/train.conllu    (raw treebank, ~46 MB)
data/processed/features/features.csv
models/pcfg/pcfg_surprisals.json
models/is_score/is_scores.json
models/xglm/xglm_surprisals.json
models/gemma-3b/gemma3_4b_surprisals.json
models/adaptive_lstm/vocab.json
```
Upload your existing `hindi_wordorder_replication` project folder (or just these specific items) to Drive, keeping the subfolder layout -- the **Path configuration** cell below defaults to that layout under one `PROJECT_ROOT`, and every path is individually overridable if yours differs.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Path configuration ──────────────────────────────────────────────────────
# Set each path to wherever that file actually lives on your Drive -- they can
# all be in different folders, you do not need to collect/rename anything.
# Example: if your original project folder was uploaded as-is to
# 'My Drive/hindi_wordorder_replication', these defaults already match it.

PROJECT_ROOT = '/content/drive/MyDrive/hindi_wordorder_replication'  # used only by the defaults below

SRC_DIR      = PROJECT_ROOT  # the folder that directly *contains* src/ (for imports)
TRAIN_CONLLU_DIR = f'{PROJECT_ROOT}/data/ud_hindi'  # folder containing train.conllu
FEATURES_CSV = f'{PROJECT_ROOT}/data/processed/features/features.csv'
PCFG_JSON    = f'{PROJECT_ROOT}/models/pcfg/pcfg_surprisals.json'
IS_JSON      = f'{PROJECT_ROOT}/models/is_score/is_scores.json'
XGLM_JSON    = f'{PROJECT_ROOT}/models/xglm/xglm_surprisals.json'
GEMMA_JSON   = f'{PROJECT_ROOT}/models/gemma-3b/gemma3_4b_surprisals.json'
VOCAB_JSON   = f'{PROJECT_ROOT}/models/adaptive_lstm/vocab.json'

# If any file is somewhere else, just overwrite its line above, e.g.:
# XGLM_JSON = '/content/drive/MyDrive/some_other_folder/xglm_surprisals.json'

required = {
    'SRC_DIR (needs a src/ subfolder)':      os.path.join(SRC_DIR, 'src'),
    'TRAIN_CONLLU_DIR (needs train.conllu)': os.path.join(TRAIN_CONLLU_DIR, 'train.conllu'),
    'FEATURES_CSV': FEATURES_CSV, 'PCFG_JSON': PCFG_JSON, 'IS_JSON': IS_JSON,
    'XGLM_JSON': XGLM_JSON, 'GEMMA_JSON': GEMMA_JSON, 'VOCAB_JSON': VOCAB_JSON,
}
missing = {name: path for name, path in required.items() if not os.path.exists(path)}
if missing:
    raise FileNotFoundError(
        'Could not find these -- edit the path variables above to match '
        f'where they actually are on your Drive: {missing}')
print('All required paths found:')
for name, path in required.items():
    print(f'  {name:<40} -> {path}')

In [ ]:
!pip install -q pandas numpy scipy scikit-learn statsmodels matplotlib

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
%matplotlib inline

sys.path.insert(0, SRC_DIR)
from src.data_loading.loader import HUTBLoader
from src.preprocessing.filter import SentenceFilter
from src.variant_generation.generator import VariantGenerator
from src.variant_generation.grammar_filter import GrammarFilter

# Validated 3-series categorical palette, fixed order (blue / orange / aqua)
C_LSTM, C_XGLM, C_GEMMA = '#2a78d6', '#eb6834', '#1baf7a'
INK, INK_MUTED, GRIDLINE = '#0b0b0b', '#898781', '#e1e0d9'

def style_axes(ax):
    for s in ('top', 'right', 'left'):
        ax.spines[s].set_visible(False)
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(left=False)

def bar_labels(ax, bars, fmt='{:.1f}'):
    for b in bars:
        h = b.get_height()
        ax.annotate(fmt.format(h), (b.get_x() + b.get_width() / 2, h),
                    textcoords='offset points', xytext=(0, 3),
                    ha='center', va='bottom', fontsize=8.5, color=INK)

## Step 1 -- Load the raw UD_Hindi-HDTB treebank (live)

In [ ]:
loader = HUTBLoader(TRAIN_CONLLU_DIR)
raw = loader.load_train()
sentences = [s['tokens'] for s in raw]
print(f'Raw sentences loaded from train.conllu: {len(raw):,}')

## Step 2 -- Filter sentences to the paper's criteria (live)

Keeps only declarative, non-negated, projective sentences with a verbal root and enough preverbal arguments -- the same `SentenceFilter` used throughout the project.

In [ ]:
filt = SentenceFilter()
filtered, rejected_df = filt.filter_with_rejection_tracking(sentences)

print(f'Passed filter: {len(filtered):,}  (paper: 1,996)')
print('Rejection breakdown:')
for reason, count in filt.stats['rejected'].items():
    if count > 0:
        print(f'  {reason:<22}: {count}')

## Step 3 -- Generate non-canonical word-order variants (live)

For each filtered sentence, permute its preverbal core dependents (subtree-preserving DFS, capped at 99 variants/sentence) and classify each as Canonical / DO-fronted / IO-fronted, same as `scripts/run_filter_and_generate.py`. Writes to local Colab disk (`/content/`), not Drive -- this run is a correctness check, not the data the rest of the notebook uses (see the note after it runs).

In [ ]:
gf = GrammarFilter()
gf.build_from_corpus(filtered)
print(f'Attested transition pairs: {len(gf.attested)}')

metadata_df = pd.DataFrame([
    {'sent_id': raw[i].get('sent_id') or f'sent_{i:04d}'}
    for i in range(len(filtered))
])

gen = VariantGenerator(max_variants=99, grammar_filter=gf)
live_variants_df = gen.generate_for_all(filtered, metadata_df, output_dir='/content/variant_demo_output')

live_refs = live_variants_df[live_variants_df['is_original'] == 1]
live_vars = live_variants_df[live_variants_df['is_original'] == 0]
print()
print(f'Live-generated reference sentences: {len(live_refs):,}  (cached dataset: 2,320)')
print(f'Live-generated variants:            {len(live_vars):,}  (cached dataset: 76,545)')
print(f'Live-generated grand total:         {len(live_variants_df):,}  (cached dataset: 78,865)')

## Step 4 -- Why trigram / PCFG / adaptive-LSTM / XGLM-1.7B / Gemma-3 come from cache

Steps 1-3 above reran the real pipeline from raw text and reproduced the exact same totals as the cached dataset (2,320 reference sentences, 76,545 variants, 78,865 total) -- that's a live correctness check, not a formality. Dependency length (`dl`) comes straight out of that generation step.

The other four features are **not** recomputed live here, each for a concrete, checkable reason:

| Feature | Why it's loaded from cache | Where it was actually computed |
|---|---|---|
| Trigram surprisal | needs a prebuilt KenLM language model trained on a Hindi Wikipedia dump (~1.4 GB of model files) | locally, `scripts/build_trigram_filtered.py` |
| IS / givenness score | cheap to compute, but cached alongside the rest for a single consistent `features.csv` | locally, `scripts/compute_is_scores.py` |
| PCFG surprisal | 5-fold CV constituency parsing with the Berkeley Parser -- needs a Java runtime, not just Python | locally, `scripts/build_pcfg_berkeley.py` |
| Adaptive-LSTM surprisal | a 2-layer/650-hidden LSTM (80.3M params, vocab 112,876 types at min\_freq=2) trained from scratch on a Hindi Wikipedia corpus; final checkpoint validation perplexity 68.35 | Kaggle (T4 GPU), `dlt-wo` notebook |
| XGLM-1.7B / Gemma-3 surprisal | 1.7B/4B-parameter LLM inference over ~78,865 variants each -- needs a GPU, ~30-45 min per model | Kaggle (T4 GPU), `scripts/kaggle_llm_scoring.py` |

One more reason the freshly-generated variants from Step 3 aren't fed into the scoring below: `VariantGenerator` uses a seeded-random fallback for sentences with 5+ reorderable arguments (too many permutations to enumerate exhaustively), and -- empirically, checked while building this notebook -- that can assign the *same total counts* but a *different `variant_id` <-> sentence mapping* across separate runs. Since the four cached score files are keyed by `variant_id` from one specific original run, Step 5 below loads that original `features.csv` (not `live_variants_df`) to guarantee every score lines up with the exact variant it was computed for.

### Reference: the actual trigram-LM training code (not executed here)

This is the real code that trained `models/trigram/hindi_trigram.arpa`/`.binary`, copied verbatim from `notebooks/MTP_prject_folder_structure.ipynb` (the Colab notebook where it was actually run) so the professor can see it without this notebook needing to run it. It is **not executed in this run** -- the `#!` prefix below marks every line as a shell/Python comment. Running it for real needs KenLM built from source, a Hindi Wikipedia dump (~700 MB), and 3-5 minutes; the resulting `.arpa`/`.binary` files are ~1.4 GB, which is why Step 4's cached `trigram_surprisals.json` is used instead.

```python
# ── Why Wikipedia instead of the paper's EMILLE corpus ──────────────────────
# The paper (Ranjan & van Schijndel 2024) trained its trigram LM on EMILLE
# (1M Hindi sentences, SRILM toolkit). The EMILLE subset available to this
# project is only the parallel/translated section (~6,884 sentences) -- 145x
# too small for a reliable trigram LM. Hindi Wikipedia is the same register
# (formal written Hindi) as HUTB newswire, UTF-8 Devanagari, freely available,
# ~700MB. KenLM (open source) replaces SRILM (proprietary) with the same
# trigram algorithm and Good-Turing-equivalent smoothing.

# ── Step 1: build KenLM from source ──────────────────────────────────────────
#!apt-get install -qq libboost-all-dev cmake build-essential
#!git clone https://github.com/kpu/kenlm.git /tmp/kenlm
#!cd /tmp/kenlm && mkdir -p build && cd build && \
#    cmake .. -DCMAKE_BUILD_TYPE=Release && \
#    make -j4 lmplz build_binary

# ── Step 2: train the trigram LM on the Hindi Wikipedia corpus ──────────────
#model_dir   = f'{PROJECT_ROOT}/models/trigram'
#corpus_path = f'{PROJECT_ROOT}/data/external/hindi_corpus_50m/hindi_sentences.txt'
#arpa_path   = f'{model_dir}/hindi_trigram.arpa'
#binary_path = f'{model_dir}/hindi_trigram.binary'
#
## order 3 = trigram, --discount_fallback = Good-Turing equivalent
#!cat "{corpus_path}" | /tmp/kenlm/build/bin/lmplz \
#    -o 3 --discount_fallback > "{arpa_path}"
#!/tmp/kenlm/build/bin/build_binary "{arpa_path}" "{binary_path}"

# ── Step 3: sanity check -- canonical order should surprise the model less ──
#import kenlm, math
#model = kenlm.Model(binary_path)
#def surprisal(text):
#    return -model.score(text, bos=True, eos=True) * math.log(10)
#sov_s = surprisal('राम ने किताब पढ़ी')   # canonical SOV
#osv_s = surprisal('किताब राम ने पढ़ी')   # non-canonical OSV
#print('SOV surprisal < OSV surprisal :', sov_s < osv_s)   # Expected: True
```

Not shown here: the earlier cells in that notebook that download the Hindi Wikipedia XML dump and sentence-split it into `hindi_sentences.txt` -- that extraction step feeds both the trigram LM above and the adaptive-LSTM's training corpus, and is the same kind of heavy, one-time, Colab/local step as everything else in the table above.

## Step 5 -- Load the authoritative scored dataset and merge the cached surprisal scores

In [ ]:
df = pd.read_csv(FEATURES_CSV)

with open(PCFG_JSON, encoding='utf-8') as f:
    pcfg = json.load(f)
with open(IS_JSON, encoding='utf-8') as f:
    is_sc = json.load(f)
with open(XGLM_JSON, encoding='utf-8') as f:
    xglm_sc = json.load(f)
with open(GEMMA_JSON, encoding='utf-8') as f:
    gemma_sc = json.load(f)
with open(VOCAB_JSON, encoding='utf-8') as f:
    w2i = json.load(f)['w2i']

df['surprisal_pcfg']   = df['variant_id'].map(pcfg).fillna(0.0)
df['is_score']         = df['variant_id'].map(is_sc).fillna(0)
df['surprisal_xglm']   = df['variant_id'].map(lambda v: xglm_sc.get(v, 100.0))
df['surprisal_gemma3'] = df['variant_id'].map(lambda v: gemma_sc.get(v, 100.0))

print(f'Variants loaded: {len(df):,}')
print(f'XGLM scores real:   {(df["surprisal_xglm"] < 100.0).sum():,}')
print(f'Gemma-3 scores real: {(df["surprisal_gemma3"] < 100.0).sum():,}')
df.head(3)

### Table 1 -- reference sentences / variants per construction type

In [ ]:
refs     = df[df['is_original'] == 1].groupby('order_type')['reference_id'].nunique()
variants = df[df['is_original'] == 0].groupby('order_type').size()
table1 = pd.DataFrame({'reference_sentences': refs, 'variants': variants}).reindex(
    ['Canonical', 'DO-fronted', 'IO-fronted'])
table1

## Step 6 -- The pairwise classifier (shared by every experiment below)

Same method as the paper: for each reference sentence, take the signed feature difference between the attested order and each generated canonical-order variant, fit a logistic regression on the pooled differences, evaluate with 10-fold stratified CV.

In [ ]:
def build_pairs(df, feat_cols, order_type):
    ref_ids = df[(df['is_original'] == 1) & (df['order_type'] == order_type)]['reference_id'].unique()
    sub = df[df['reference_id'].isin(ref_ids)]
    Xs, ys = [], []
    for _, grp in sub.groupby('reference_id'):
        rf = grp[grp['is_original'] == 1]
        vr = grp[(grp['is_original'] == 0) & (grp['order_type'] == 'Canonical')]
        if len(rf) == 0 or len(vr) == 0:
            continue
        rfv = rf.iloc[0][feat_cols].values.astype(float)
        for _, v in vr.iterrows():
            vfv = v[feat_cols].values.astype(float)
            if np.any(np.isnan(rfv)) or np.any(np.isnan(vfv)):
                continue
            Xs.append(rfv - vfv); ys.append(1)
            Xs.append(vfv - rfv); ys.append(0)
    if not Xs:
        return None, None
    return np.array(Xs), np.array(ys)

def run_cv(X, y, n_folds=10, seed=42):
    if X is None or len(X) == 0:
        return float('nan')
    cv, accs = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed), []
    for tr, te in cv.split(X, y):
        sc = StandardScaler()
        Xtr, Xte = sc.fit_transform(X[tr]), sc.transform(X[te])
        clf = LogisticRegression(max_iter=1000, random_state=seed)
        clf.fit(Xtr, y[tr])
        accs.append(accuracy_score(y[te], clf.predict(Xte)))
    return np.mean(accs) * 100

print('build_pairs() and run_cv() defined.')

## Step 7 -- Replication check: Table 2

Our 5-feature (adaptive-LSTM) classifier, run live here, vs. the numbers Ranjan & van Schijndel (2024) report in their paper (hardcoded only because they're *external* published numbers we're comparing against -- not something this pipeline produces).

In [ ]:
LSTM_FEATS = ['dl', 'is_score', 'surprisal_pcfg', 'surprisal_trigram', 'surprisal_adaptive_lstm']

X_do, y_do = build_pairs(df, LSTM_FEATS, 'DO-fronted')
X_io, y_io = build_pairs(df, LSTM_FEATS, 'IO-fronted')
acc_do = run_cv(X_do, y_do)
acc_io = run_cv(X_io, y_io)

PAPER_ACC = {'DO-fronted': 80.46, 'IO-fronted': 90.02}  # Ranjan & van Schijndel (2024), published

table2 = pd.DataFrame({
    'Construction': ['DO-fronted', 'IO-fronted'],
    'Our accuracy (%)':   [round(acc_do, 2), round(acc_io, 2)],
    'Paper accuracy (%)': [PAPER_ACC['DO-fronted'], PAPER_ACC['IO-fronted']],
})
table2['Diff (pts)'] = (table2['Our accuracy (%)'] - table2['Paper accuracy (%)']).round(2)
table2

## Step 8 -- LLM extension: Table 3 and Figure 1

Single-feature ablations plus the full 5-feature model with the surprisal source swapped between adaptive-LSTM, XGLM-1.7B and Gemma-3.

In [ ]:
BASE_FEATS  = ['dl', 'is_score', 'surprisal_pcfg', 'surprisal_trigram']
XGLM_FEATS  = BASE_FEATS + ['surprisal_xglm']
GEMMA_FEATS = BASE_FEATS + ['surprisal_gemma3']

experiments = [
    ('Trigram only',               ['surprisal_trigram']),
    ('Dep. length only',           ['dl']),
    ('IS score only',              ['is_score']),
    ('Adaptive LSTM only',         ['surprisal_adaptive_lstm']),
    ('XGLM only',                  ['surprisal_xglm']),
    ('Gemma-3 only',               ['surprisal_gemma3']),
    ('All 5 (adaptive LSTM)',      LSTM_FEATS),
    ('LSTM -> XGLM replacement',   XGLM_FEATS),
    ('LSTM -> Gemma-3 replacement', GEMMA_FEATS),
]

rows = []
for label, feats in experiments:
    row = {'Experiment': label}
    for ot in ['DO-fronted', 'IO-fronted']:
        X, y = build_pairs(df, feats, ot)
        row[ot] = round(run_cv(X, y), 1)
    rows.append(row)
    print(f'  {label:<30} DO={row["DO-fronted"]:.1f}%  IO={row["IO-fronted"]:.1f}%')

table3 = pd.DataFrame(rows)
table3

In [ ]:
# Figure 1 -- full 5-feature model accuracy by construction, 3 surprisal sources
full_rows = {
    'Adaptive LSTM\n(paper\'s setup)': 'All 5 (adaptive LSTM)',
    'LSTM -> XGLM-1.7B':               'LSTM -> XGLM replacement',
    'LSTM -> Gemma-3':                 'LSTM -> Gemma-3 replacement',
}
t3 = table3.set_index('Experiment')

fig, ax = plt.subplots(figsize=(6, 3.4))
x = np.arange(2)
width = 0.26
colors = [C_LSTM, C_XGLM, C_GEMMA]
for i, (model_label, exp_label) in enumerate(full_rows.items()):
    vals = [t3.loc[exp_label, 'DO-fronted'], t3.loc[exp_label, 'IO-fronted']]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=model_label, color=colors[i],
                  edgecolor='white', linewidth=0.5)
    bar_labels(ax, bars)

ax.set_xticks(x); ax.set_xticklabels(['DO-fronted', 'IO-fronted'])
ax.set_ylabel('Classification accuracy (%)')
ax.set_ylim(0, 100)
style_axes(ax)
ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8.5)
ax.set_title('Figure 1 -- Accuracy by construction')
fig.tight_layout()
plt.show()

## Step 9 -- Coefficient significance: Table 4 and Figure 2

Fits the full 5-feature GLM (statsmodels, so we get standard errors / Z / VIF, not just CV accuracy) for each of the 3 surprisal sources, for both constructions.

In [ ]:
def glm_coefs(df, feat_cols, feat_labels, order_type):
    ref_ids = df[(df['is_original'] == 1) & (df['order_type'] == order_type)]['reference_id'].unique()
    sub = df[df['reference_id'].isin(ref_ids)]
    Xs, ys = [], []
    for _, grp in sub.groupby('reference_id'):
        rf = grp[grp['is_original'] == 1]
        vr = grp[(grp['is_original'] == 0) & (grp['order_type'] == 'Canonical')]
        if len(rf) == 0 or len(vr) == 0:
            continue
        rfv = rf.iloc[0][feat_cols].values.astype(float)
        for _, v in vr.iterrows():
            vfv = v[feat_cols].values.astype(float)
            if np.any(np.isnan(rfv)) or np.any(np.isnan(vfv)):
                continue
            Xs.append(rfv - vfv); ys.append(1)
            Xs.append(vfv - rfv); ys.append(0)
    X_raw, y = np.array(Xs), np.array(ys)
    X_sc = StandardScaler().fit_transform(X_raw)
    X_c  = sm.add_constant(X_sc)
    res  = sm.Logit(y, X_c).fit(disp=0, maxiter=300)
    vifs = [variance_inflation_factor(X_c, i) for i in range(X_c.shape[1])]
    labels = ['Intercept'] + feat_labels
    return pd.DataFrame({
        'Predictor': labels, 'Estimate': res.params, 'Std_Error': res.bse,
        'Z_value': res.tvalues, 'P_value': res.pvalues, 'VIF': vifs,
    })

BASE_LABELS = ['Dependency length', 'IS score', 'PCFG surprisal', 'Trigram surprisal']
models = {
    'Adaptive LSTM': (LSTM_FEATS,  BASE_LABELS + ['Adaptive LSTM surprisal']),
    'XGLM-1.7B':     (XGLM_FEATS,  BASE_LABELS + ['XGLM surprisal']),
    'Gemma-3':       (GEMMA_FEATS, BASE_LABELS + ['Gemma-3 surprisal']),
}

coef_rows = []
for construction in ['DO-fronted', 'IO-fronted']:
    for model_name, (feats, labels) in models.items():
        res = glm_coefs(df, feats, labels, construction)
        surprisal_row = res.iloc[-1]  # the model-specific surprisal predictor is always last
        coef_rows.append({
            'Construction': construction, 'Model': model_name,
            'Estimate': round(surprisal_row['Estimate'], 3),
            '|Z|': round(abs(surprisal_row['Z_value']), 2),
            'P_value': surprisal_row['P_value'],
        })
        print(f'  {construction:<12} {model_name:<14} Estimate={surprisal_row["Estimate"]:+.3f}  |Z|={abs(surprisal_row["Z_value"]):.2f}')

table4 = pd.DataFrame(coef_rows)
table4

In [ ]:
# Figure 2 -- coefficient magnitude (a) and |Z|-statistic (b)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(7.5, 3.4))
constructions = ['DO-fronted', 'IO-fronted']
x = np.arange(2)
width = 0.26
model_color = {'Adaptive LSTM': C_LSTM, 'XGLM-1.7B': C_XGLM, 'Gemma-3': C_GEMMA}

for i, model_name in enumerate(model_color):
    sub = table4[table4['Model'] == model_name].set_index('Construction')
    est  = [abs(sub.loc[c, 'Estimate']) for c in constructions]
    zval = [sub.loc[c, '|Z|'] for c in constructions]
    offs = (i - 1) * width
    b1 = axL.bar(x + offs, est,  width, label=model_name, color=model_color[model_name], edgecolor='white', linewidth=0.5)
    b2 = axR.bar(x + offs, zval, width, label=model_name, color=model_color[model_name], edgecolor='white', linewidth=0.5)
    bar_labels(axL, b1); bar_labels(axR, b2)

for ax, title, ylabel in [(axL, '(a) Coefficient magnitude', '|Estimate|'), (axR, '(b) |Z|-statistic', '|Z|')]:
    ax.set_xticks(x); ax.set_xticklabels(constructions)
    ax.set_title(title, fontsize=10); ax.set_ylabel(ylabel)
    style_axes(ax)
axR.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=8.5)
fig.suptitle('Figure 2 -- Coefficient magnitude and |Z|-statistic')
fig.tight_layout()
plt.show()

## Step 10 -- Why do LLMs win? Table 5 and Figure 3

Per-sentence: does the LSTM surprisal correctly rank the attested order above the mean of its canonical-order variants, and does XGLM? Stratified by dependency length and by OOV rate against the LSTM's own vocabulary.

In [ ]:
records = []
for order_type in ['DO-fronted', 'IO-fronted']:
    corpus_rows = df[(df['is_original'] == 1) & (df['order_type'] == order_type)]
    for _, corp in corpus_rows.iterrows():
        cands = df[(df['reference_id'] == corp['reference_id']) &
                   (df['is_original'] == 0) & (df['order_type'] == 'Canonical')]
        if len(cands) == 0:
            continue
        lstm_correct = corp['surprisal_adaptive_lstm'] < cands['surprisal_adaptive_lstm'].mean()
        xglm_correct = corp['surprisal_xglm'] < cands['surprisal_xglm'].mean()
        words = str(corp['text']).split()
        oov = sum(1 for w in words if w not in w2i) / len(words) if words else np.nan
        records.append({'dl': corp['dl'], 'oov_rate': oov,
                        'lstm_correct': lstm_correct, 'xglm_correct': xglm_correct})

err_df = pd.DataFrame(records)
print(f'Per-sentence comparisons: {len(err_df)}')

def acc_row(mask, label):
    sub = err_df[mask]
    return {'Subset': label, 'n': len(sub),
            'LSTM acc. (%)': round(100 * sub['lstm_correct'].mean(), 1),
            'XGLM acc. (%)': round(100 * sub['xglm_correct'].mean(), 1)}

table5 = pd.DataFrame([
    acc_row(err_df['dl'] > 50, 'Dep. length > 50'),
    acc_row(err_df['dl'] <= 50, 'Dep. length <= 50'),
    acc_row(err_df['oov_rate'] > 0.1, 'OOV rate > 0.1'),
    acc_row(err_df['oov_rate'] <= 0.1, 'OOV rate <= 0.1'),
    acc_row(pd.Series(True, index=err_df.index), 'Overall'),
])
table5

In [ ]:
# Figure 3 -- stratified accuracy by dependency length (a) and OOV rate (b)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(7.5, 3.4))
width = 0.32
x = np.arange(2)

t5 = table5.set_index('Subset')
dl_labels  = [f'DL > 50\n(n={t5.loc["Dep. length > 50", "n"]})', f'DL <= 50\n(n={t5.loc["Dep. length <= 50", "n"]})']
oov_labels = [f'OOV > 0.1\n(n={t5.loc["OOV rate > 0.1", "n"]})', f'OOV <= 0.1\n(n={t5.loc["OOV rate <= 0.1", "n"]})']

b1 = axL.bar(x - width/2, [t5.loc['Dep. length > 50', 'LSTM acc. (%)'], t5.loc['Dep. length <= 50', 'LSTM acc. (%)']], width, label='Adaptive LSTM', color=C_LSTM, edgecolor='white', linewidth=0.5)
b2 = axL.bar(x + width/2, [t5.loc['Dep. length > 50', 'XGLM acc. (%)'], t5.loc['Dep. length <= 50', 'XGLM acc. (%)']], width, label='XGLM-1.7B', color=C_XGLM, edgecolor='white', linewidth=0.5)
bar_labels(axL, b1); bar_labels(axL, b2)
axL.set_xticks(x); axL.set_xticklabels(dl_labels)
axL.set_title('(a) By dependency length', fontsize=10); axL.set_ylabel('Accuracy (%)')

b3 = axR.bar(x - width/2, [t5.loc['OOV rate > 0.1', 'LSTM acc. (%)'], t5.loc['OOV rate <= 0.1', 'LSTM acc. (%)']], width, label='Adaptive LSTM', color=C_LSTM, edgecolor='white', linewidth=0.5)
b4 = axR.bar(x + width/2, [t5.loc['OOV rate > 0.1', 'XGLM acc. (%)'], t5.loc['OOV rate <= 0.1', 'XGLM acc. (%)']], width, label='XGLM-1.7B', color=C_XGLM, edgecolor='white', linewidth=0.5)
bar_labels(axR, b3); bar_labels(axR, b4)
axR.set_xticks(x); axR.set_xticklabels(oov_labels)
axR.set_title('(b) By OOV rate', fontsize=10)

for ax in (axL, axR):
    ax.set_ylim(0, 100); style_axes(ax)
axR.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=8.5)
fig.suptitle('Figure 3 -- Why do LLMs win? Stratified accuracy')
fig.tight_layout()
plt.show()